# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RayyanA24/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*
## Paper Finding 1

The paper suggests that search performance signals can help identify pages that are good candidates for content refresh.

**Methodology question:**  
The label is derived from observed search performance. This is useful for prioritization, but it may not capture all reasons why a page's performance changes. External factors such as seasonality or algorithm updates could also influence results.

---

## Paper Finding 2

The paper reports that combining multiple search signals improves prioritization compared with simple rules.

**Methodology question:**  
This finding appears reasonable, but the validation should use an honest train-test split so that performance is measured on unseen data. This helps ensure the reported improvement generalizes beyond the training data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*
## Honest Validation

I compared my model using the original random train-test split and an alternative validation split. This provides a more realistic estimate of model performance and helps identify whether the model generalizes to unseen data.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import sys
import subprocess
import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git","clone","--depth","1",REPO_URL,REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

features = [
    "search_volume",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "content_age_days",
    "days_since_last_update"
]

X = df[features].fillna(0)
y = (df["trend_direction"] == "down").astype(int)

# Original split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

before = accuracy_score(y_test, pred)

# Alternative honest split
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X, y,
    test_size=0.30,
    random_state=123,
    stratify=y
)

model.fit(X_train2, y_train2)

pred2 = model.predict(X_test2)

after = accuracy_score(y_test2, pred2)

print(pd.DataFrame({
    "Validation": ["Original Split", "Alternative Split"],
    "Accuracy": [before, after]
}))


          Validation  Accuracy
0     Original Split  0.631667
1  Alternative Split  0.643333


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*
## Leakage Audit

I reviewed the final feature set and confirmed that the prediction target (`trend_direction`) and related outcome fields (such as `trend_pct`) are excluded from the model inputs. Only information that would be available before making a prediction is used.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded = [
    "trend_direction",
    "trend_pct",
    "client_id",
    "content_id"
]

print("Excluded columns:")

for c in excluded:
    print("-", c)


Excluded columns:
- trend_direction
- trend_pct
- client_id
- content_id


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*
## Claim Rewrite

**Original claim:**  
The model accurately identifies pages that need refreshing.

**Rewritten claim:**  
The model identifies pages that may benefit from content review based on observed historical search signals. The results should be interpreted as decision-support rather than proof that a page requires updating.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.